# Unsloth Deep Dive #00 — 2026 Migration Notları

Eski blog/Colab'lardan yeni kod yazıyorsan değişen noktalar.

**Not:** Bu notebook **resmi 2026 Unsloth notebook'ları** referans alınarak yazılmıştır:
- `Qwen3_(4B)_Instruct.ipynb`, `Qwen3_(4B)_Thinking.ipynb`
- `Gemma4_(E2B)_Text.ipynb`
- `Qwen3_5_(2B)_Vision.ipynb`

---

## Büyük Resim — Beş Major Değişim

| # | Değişim | Etki |
|---|---------|------|
| 1 | **`unsloth.chat_templates` modülü** | `get_chat_template`, `standardize_data_formats`, `train_on_responses_only` resmi pattern oldu |
| 2 | **`FastModel` (yeni unified API)** | Gemma 4 ve VLM'lerde tercih ediliyor; `FastLanguageModel` Qwen3 için hala recommended |
| 3 | **`FastVisionModel` zorunlu (vision SFT)** | `UnslothVisionDataCollator` ile birlikte |
| 4 | **Dynamic 2.0 quantization** | `unsloth-bnb-4bit` suffix bazı modellerde mevcut (her modelde değil) |
| 5 | **Yeni model variants** | Qwen3-4B-Instruct-2507, Thinking-2507; Gemma 4 E/26B-A4B; Qwen3.5 multimodal |

## Değişim 1 — `unsloth.chat_templates` Modülü

**En kritik değişiklik.** Önceki rehberlerin çoğu `assistant_only_loss=True` (TRL native) öneriyordu. Resmi Unsloth pattern'i farklı:

```python
from unsloth.chat_templates import (
    get_chat_template,           # Tokenizer'a chat template uygula
    standardize_data_formats,    # Dataset format normalize
    train_on_responses_only,     # Loss masking — Unsloth'un yöntemi
)
```

### Eski (TRL native)
```python
trainer = SFTTrainer(
    ...
    args = SFTConfig(
        assistant_only_loss = True,        # TRL'nin yöntemi
    ),
)
```

**Sorunlar:**
- Chat template `{% generation %}` keyword'ü gerektiriyor
- VLM modellerde patlar (`_is_vlm` kontrolü)
- Bazı modellerde tüm labels = -100 hatası

### Yeni (Unsloth resmi)
```python
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(...)  # önce normal trainer

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)
```

**Avantajlar:**
- Manuel token belirtme — chat template'e bağımlı değil
- VLM kontrolünden bağımsız
- Her modelde çalışır

### Token'lar Model-Spesifik

| Model | instruction_part | response_part |
|-------|------------------|---------------|
| Qwen3 (Instruct/Thinking) | `<\|im_start\|>user\n` | `<\|im_start\|>assistant\n` |
| Gemma 4 | `<\|turn>user\n` | `<\|turn>model\n` |
| Llama 3 | `<\|start_header_id\|>user<\|end_header_id\|>\n\n` | `<\|start_header_id\|>assistant<\|end_header_id\|>\n\n` |

## Değişim 2 — `FastModel` (Yeni Unified API)

**Önemli düzeltme:** Önceki iddiamın aksine `FastLanguageModel` LEGACY DEĞİL — Qwen3 4B Instruct ve Thinking resmi notebook'ları bunu kullanıyor.

### Hiyerarşi (server'da test edilmiş)

```
FastBaseModel
  ├── FastModel              ← YENI unified API (Gemma 4, VLM text-only)
  │   ├── FastTextModel      ← FastModel'in text version'u
  │   └── FastVisionModel    ← Vision SFT için
  └── FastLlamaModel
      └── FastLanguageModel  ← Qwen3, Llama, Mistral için (HALA recommended)
```

### Ne Zaman Hangisini?

| Senaryo | Resmi Notebook'taki Class |
|---------|----------------------------|
| Qwen3 (Instruct/Thinking) text SFT | **`FastLanguageModel`** |
| Gemma 4 E2B text SFT | **`FastModel`** |
| Qwen3.5 Vision SFT | **`FastVisionModel`** |
| Llama 3 / Mistral text SFT | `FastLanguageModel` (genel kabul) |

İkisi de aktif olarak kullanılıyor — "yeni tek doğru" yok.

## Değişim 3 — Vision SFT için `UnslothVisionDataCollator`

Vision SFT'de **standart `SFTTrainer` çalışmaz**. Ekstra config + collator gerekir:

```python
from unsloth.trainer import UnslothVisionDataCollator

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),  # ZORUNLU
    train_dataset = converted_dataset,
    args = SFTConfig(
        # Vision için ZORUNLU üçlü:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
        ...
    ),
)
```

**Önemli:** Vision SFT'de `train_on_responses_only` KULLANILMAZ — collator masking'i kendi yapar.

## Değişim 4 — Dynamic 2.0 Quantization

Hub'da iki tip 4-bit varyant:

- `model-bnb-4bit` — standart bitsandbytes
- `model-unsloth-bnb-4bit` — Dynamic 2.0 (outlier-aware, ~%2-5 daha iyi)

### Önemli: Her Modelde Yok

```python
# Var mı kontrol et
from huggingface_hub import list_models
for m in list_models(author="unsloth", search="YOUR_MODEL"):
    print(m.id)
```

**Mevcut durumlar (Mayıs 2026):**
- Qwen3 (text) → `unsloth-bnb-4bit` VAR
- Gemma 4 E2B/E4B → `unsloth-bnb-4bit` VAR
- **Qwen3.5 (VLM) → SADECE full + GGUF, bnb-4bit YOK!**

Qwen3.5 için `load_in_4bit=True` runtime quantize işin görür ama Dynamic 2.0 avantajı yok.

### Resmi Notebook'larda

İlginç bulgu: Resmi notebook'ların çoğu **full precision modelden** başlıyor (`unsloth/Qwen3-4B-Instruct-2507`), pre-quantized varyantları sadece comment'te listeliyor. Yani Dynamic 2.0 bir opsiyon, default değil.

## Değişim 5 — Yeni Model Variants

### Qwen3 — `2507` versiyonları
- `unsloth/Qwen3-4B-Instruct-2507`
- `unsloth/Qwen3-4B-Thinking-2507`
- Eski Qwen3 sürümlerinin updated release'leri

### Gemma 4 — Yeni adlandırma
- `unsloth/gemma-4-E2B`, `gemma-4-E4B` — Elastic (multimodal capable)
- `unsloth/gemma-4-26B-A4B` — MoE (26B toplam, 4B aktif)
- `unsloth/gemma-4-31B` — Dense

### Qwen3.5 — Multimodal Family
- 0.8B, 2B, 4B, 9B, 27B (dense)
- 35B-A3B, 122B-A10B (MoE)
- Hepsi `Qwen3_5ForConditionalGeneration` (VLM mimari)

### Thinking Modelleri Özel — `enable_thinking`

```python
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
    enable_thinking = False,        # YENİ — reasoning modu kapat/aç
)
```

Training'te tüm veride thinking blokları, inference'ta opsiyonel kapatılabilir.

## Değişen Default Değerler

Resmi notebook'ların kullandığı ama benim önceki rehberimde yanlış olan değerler:

| Param | Yanlış (eski rehberim) | Doğru (resmi notebook) |
|-------|------------------------|------------------------|
| `weight_decay` | 0.01 | **0.001** |
| `lr_scheduler_type` | `"cosine"` | **`"linear"`** |
| `warmup` | `warmup_ratio=0.03` | **`warmup_steps=5`** |
| `random_state` | 42 | **3407** (Unsloth'un standart seed'i) |
| `processing_class=` | (TRL 1.x) | **`tokenizer=`** (resmi notebook'larda) |

### `[NEW!]` Flag'ler (Resmi notebook'larda comment olarak)

- `load_in_8bit = False` — "A bit more accurate, uses 2x memory"
- `full_finetuning = False` — "We have full finetuning now!"
- `use_gradient_checkpointing = "unsloth"` — "30% less VRAM, fits 2x larger batch"

## Migration Checklist

### 1. `unsloth.chat_templates` import et
```python
from unsloth.chat_templates import (
    get_chat_template, standardize_data_formats, train_on_responses_only,
)
```

### 2. Chat template uygula
```python
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")
```

### 3. Dataset normalize
```python
dataset = standardize_data_formats(dataset)
```

### 4. Formatting func ile text alanı oluştur
```python
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    return {"text": [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in convos]}
dataset = dataset.map(formatting_prompts_func, batched=True)
```

### 5. Trainer'ı `train_on_responses_only` ile sar
```python
trainer = SFTTrainer(...)  # `assistant_only_loss=True` KULLANMA
trainer = train_on_responses_only(trainer, instruction_part=..., response_part=...)
```

### 6. Resmi default değerlere geç
```python
SFTConfig(
    ...,
    weight_decay = 0.001,
    lr_scheduler_type = "linear",
    warmup_steps = 5,
    seed = 3407,
)
```

### 7. Inference'da `add_generation_prompt=True` unutma
```python
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
```

## Özet

Resmi Unsloth notebook'larından bilmen gereken minimum:

✅ `unsloth.chat_templates` modülü — `get_chat_template`, `standardize_data_formats`, `train_on_responses_only` 
✅ `assistant_only_loss=True` YOK — `train_on_responses_only(trainer, instruction_part, response_part)` 
✅ `FastLanguageModel` (Qwen3), `FastModel` (Gemma 4), `FastVisionModel` (Vision) — context'e göre 
✅ Vision SFT için `UnslothVisionDataCollator` + zorunlu config üçlüsü 
✅ `weight_decay=0.001`, `lr_scheduler_type="linear"`, `warmup_steps=5`, `seed=3407` 
✅ Token'lar model-spesifik: Qwen3 `<|im_start|>`, Gemma `<|turn>`, Llama `<|start_header_id|>` 

**Sıradaki notebook:** `01_internals.ipynb` — Triton kernel'lar, patching, internals.